# Dual-model hybrid evaluation (HAABSA4GCN / HAABSA7GCN)

Computes the final **ontology + neural-backup hybrid** accuracy for both models on
SemEval-2015 and SemEval-2016 **without running any model**.

The hybrid pipeline routes each test instance either to the rule-based ontology
(conclusive cases) or to the neural backup (inconclusive cases, listed in
`rem<year>.csv`). Reported hybrid accuracy is the instance-count-weighted average
of the two stages:

$$\text{hybrid} = \frac{n_\text{conclusive}\cdot \text{ont\_acc} + n_\text{backup}\cdot \text{backup\_acc\_on\_remainder}}{n_\text{total}}$$

## Inputs
- `eval_files/{4gcn,7gcn}_{2015,2016}.txt` — backup predictions at the
  validation-selected epoch of each rank-1 run (seed 7). Fingerprint-verified
  against each run's JSON on both test acc and macro F1. Format:
  `gold <TAB> pred <TAB> sentence <TAB> aspect`, labels `0=neg 1=neu 2=pos`.
- `../data/rem{2015,2016}.csv` — **canonical** indices of test instances the
  ontology deferred to the backup (do-not-modify list; identical across all four
  source zips).
- ontology-accuracy constants — hardcoded below (re-measured macOS port).


> Run from `src/` (the working directory of this notebook).

In [1]:
import numpy as np
import pandas as pd

ONT_ACC = {2015: 0.813559, 2016: 0.876263}
N_TOTAL = {2015: 597, 2016: 650}

EVAL = {
    ("4GCN", 2015): "eval_files/4gcn_2015.txt",
    ("4GCN", 2016): "eval_files/4gcn_2016.txt",
    ("7GCN", 2015): "eval_files/7gcn_2015.txt",
    ("7GCN", 2016): "eval_files/7gcn_2016.txt",
}
# rem indices:
REM = {2015: "../data/rem2015.csv", 2016: "../data/rem2016.csv"}

In [2]:
def load_preds(path):
    """Return (n, 2) array of (gold, pred) ints from a backup eval file."""
    rows = []
    for line in open(path, encoding="utf-8"):
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 2:
            continue
        rows.append((int(parts[0]), int(parts[1])))
    return np.array(rows)


def hybrid_accuracy(model, year, verbose=True):
    n_total = N_TOTAL[year]
    ont_acc = ONT_ACC[year]
    backup_idx = np.loadtxt(REM[year], dtype=int)
    n_backup = len(backup_idx)
    n_conclusive = n_total - n_backup

    pred = load_preds(EVAL[(model, year)])
    assert len(pred) == n_total, (
        f"{model} {year}: eval file has {len(pred)} rows, expected {n_total}"
    )

    gold, p = pred[:, 0], pred[:, 1]
    acc_full = (gold == p).mean()                            # backup-only, full test set
    acc_backup = (gold[backup_idx] == p[backup_idx]).mean()  # backup on deferred subset
    acc_hybrid = (n_conclusive * ont_acc + n_backup * acc_backup) / n_total

    if verbose:
        print(f"=== {model}  SemEval-{year} ===")
        print(f"  backup-only (full test set)   : {100*acc_full:.2f}%")
        print(f"  backup on deferred subset     : {100*acc_backup:.2f}%   "
              f"({n_backup} of {n_total} instances)")
        print(f"  ontology on conclusive subset : {100*ont_acc:.2f}%   "
              f"({n_conclusive} instances)")
        print(f"  HYBRID (HAABSA{model[0]}GCN)         : {100*acc_hybrid:.2f}%")
        print()
    return {"model": model, "year": year, "acc_full": acc_full,
            "acc_backup": acc_backup, "acc_hybrid": acc_hybrid}

In [3]:
results = []
for model in ("4GCN", "7GCN"):
    for year in (2015, 2016):
        results.append(hybrid_accuracy(model, year))

=== 4GCN  SemEval-2015 ===
  backup-only (full test set)   : 82.08%
  backup on deferred subset     : 82.45%   (302 of 597 instances)
  ontology on conclusive subset : 81.36%   (295 instances)
  HYBRID (HAABSA4GCN)         : 81.91%

=== 4GCN  SemEval-2016 ===
  backup-only (full test set)   : 89.08%
  backup on deferred subset     : 89.76%   (254 of 650 instances)
  ontology on conclusive subset : 87.63%   (396 instances)
  HYBRID (HAABSA4GCN)         : 88.46%

=== 7GCN  SemEval-2015 ===
  backup-only (full test set)   : 81.74%
  backup on deferred subset     : 81.79%   (302 of 597 instances)
  ontology on conclusive subset : 81.36%   (295 instances)
  HYBRID (HAABSA7GCN)         : 81.57%

=== 7GCN  SemEval-2016 ===
  backup-only (full test set)   : 89.85%
  backup on deferred subset     : 90.94%   (254 of 650 instances)
  ontology on conclusive subset : 87.63%   (396 instances)
  HYBRID (HAABSA7GCN)         : 88.92%



In [4]:
# Tidy summary tables
df = pd.DataFrame(results)

hybrid_tbl = (df.pivot(index="model", columns="year", values="acc_hybrid") * 100).round(2)
backup_tbl = (df.pivot(index="model", columns="year", values="acc_full") * 100).round(2)

print("Hybrid accuracy, % (Table 2):")
display(hybrid_tbl)
print("Backup-only accuracy, % (Tables 7/8):")
display(backup_tbl)

Hybrid accuracy, % (Table 2):


year,2015,2016
model,,
4GCN,81.91,88.46
7GCN,81.57,88.92


Backup-only accuracy, % (Tables 7/8):


year,2015,2016
model,,
4GCN,82.08,89.08
7GCN,81.74,89.85
